# Week 7 workshop · Particle swarm optimisation

<span class="workshop-download-enabled" aria-hidden="true"></span>


# Context


Particle swarm optimisation uses personal and globally shared records to guide a population through a solution space. This workshop investigates how the weight given to shared information changes the behaviour of the search.

> How does the weight given to globally shared information affect the reliability, solution quality and concentration of a particle swarm search?


# Specify the PSO investigation

## PSO baseline

For a minimisation problem, particle $i$ stores a candidate $\mathbf{x}_i^n$, a displacement $\mathbf{v}_i^n$ and its personal-best position $\mathbf{p}_i$. Every particle can use the best-so-far position $\mathbf{g}$ shared globally.

At iteration $n$,

$$
\mathbf v_i^{n+1}=w\mathbf v_i^n
+c_1\mathbf r_{1,i}^n\odot[\mathbf p_i-\mathbf x_i^n]
+c_2\mathbf r_{2,i}^n\odot[\mathbf g-\mathbf x_i^n],
$$

followed by $\mathbf{x}_i^{n+1}=\mathbf{x}_i^n+\mathbf{v}_i^{n+1}$. Here $\odot$ denotes componentwise multiplication. The coefficient $w$ is inertia, while $c_1$ and $c_2$ weight personal and shared information. Every component of $\mathbf r_{1,i}^n$ and $\mathbf r_{2,i}^n$ is an independent uniform draw from $[0,1]$.

Initial positions are uniform within the feasible bounds. Each component of the initial displacement is drawn uniformly from plus or minus 10% of that coordinate's range. This allows particles to search when $c_2=0$. The supplied boundary rule clips positions to the bounds and retains the calculated displacement.


# Validate the implementation

The following two cells provide the PSO engine. Treat it as inherited code: identify its assumptions and establish that it behaves as specified before using it for an investigation.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

def objective_landscape(x):
    x = np.asarray(x, dtype=float)
    dx = x[..., 0] - 1.3
    dy = x[..., 1] + 0.8
    return (
        0.035 * (dx**2 + 1.15 * dy**2)
        + (np.sin(1.15 * dx) + 0.42 * np.sin(2.05 * dy))**2
        + 0.65 * (np.sin(0.78 * dy)
                  + 0.38 * np.sin(2.35 * dx + 0.2 * dy))**2
    )


In [2]:
def initialise_swarm(n_particles, n_dims, bounds, objective, rng,
                     initial_velocity_fraction=0.10):
    if not isinstance(n_particles, (int, np.integer)) or n_particles < 1:
        raise ValueError("n_particles must be a positive integer")
    if not isinstance(n_dims, (int, np.integer)) or n_dims < 1:
        raise ValueError("n_dims must be a positive integer")
    if initial_velocity_fraction < 0:
        raise ValueError("initial_velocity_fraction must be non-negative")

    lower = np.broadcast_to(np.asarray(bounds[0], dtype=float), (n_dims,))
    upper = np.broadcast_to(np.asarray(bounds[1], dtype=float), (n_dims,))
    if not np.all(np.isfinite(lower)) or not np.all(np.isfinite(upper)):
        raise ValueError("bounds must be finite")
    if np.any(upper <= lower):
        raise ValueError("every upper bound must exceed its lower bound")

    positions = rng.uniform(lower, upper, size=(n_particles, n_dims))
    velocity_limit = initial_velocity_fraction * (upper - lower)
    velocities = rng.uniform(-velocity_limit, velocity_limit,
                             size=(n_particles, n_dims))
    personal_best = positions.copy()
    personal_scores = np.asarray(objective(personal_best), dtype=float)
    if personal_scores.shape != (n_particles,):
        raise ValueError("objective must return one score per particle")
    if not np.all(np.isfinite(personal_scores)):
        raise ValueError("objective returned a non-finite score")
    return positions, velocities, personal_best, personal_scores


def pso_step(positions, velocities, personal_best, personal_scores,
             objective, bounds, rng, inertia=0.72,
             personal_weight=1.49, shared_weight=1.49, max_speed=None):
    if max_speed is not None and max_speed <= 0:
        raise ValueError("max_speed must be positive when supplied")

    global_best = personal_best[np.argmin(personal_scores)].copy()
    r_personal = rng.random(positions.shape)
    r_shared = rng.random(positions.shape)
    velocities = (
        inertia * velocities
        + personal_weight * r_personal * (personal_best - positions)
        + shared_weight * r_shared * (global_best - positions)
    )
    if max_speed is not None:
        speed = np.linalg.norm(velocities, axis=1, keepdims=True)
        velocities *= np.minimum(1.0, max_speed / np.maximum(speed, 1e-12))

    # Boundary rule: clip the position and retain the calculated velocity.
    positions = np.clip(positions + velocities, bounds[0], bounds[1])
    scores = np.asarray(objective(positions), dtype=float)
    if scores.shape != personal_scores.shape:
        raise ValueError("objective must return one score per particle")
    if not np.all(np.isfinite(scores)):
        raise ValueError("objective returned a non-finite score")

    personal_best = personal_best.copy()
    personal_scores = personal_scores.copy()
    improved = scores < personal_scores
    personal_best[improved] = positions[improved]
    personal_scores[improved] = scores[improved]
    return positions, velocities, personal_best, personal_scores, scores


def run_pso(objective=objective_landscape, seed=7, n_particles=30, n_dims=2,
            bounds=((-6.0, -5.0), (6.0, 5.0)), steps=100,
            initial_velocity_fraction=0.10, **update_parameters):
    if not isinstance(steps, (int, np.integer)) or steps < 0:
        raise ValueError("steps must be a non-negative integer")

    rng = np.random.default_rng(seed)
    initial = initialise_swarm(
        n_particles, n_dims, bounds, objective, rng,
        initial_velocity_fraction=initial_velocity_fraction,
    )
    positions, velocities, personal_best, personal_scores = initial

    position_history = [positions.copy()]
    best_scores = [personal_scores.min()]
    diversity = [np.mean(np.linalg.norm(
        positions - positions.mean(axis=0), axis=1
    ))]

    for _ in range(steps):
        positions, velocities, personal_best, personal_scores, _ = pso_step(
            positions, velocities, personal_best, personal_scores,
            objective, bounds, rng, **update_parameters,
        )
        position_history.append(positions.copy())
        best_scores.append(personal_scores.min())
        diversity.append(np.mean(np.linalg.norm(
            positions - positions.mean(axis=0), axis=1
        )))

    best_index = int(np.argmin(personal_scores))
    return {
        "positions": np.asarray(position_history),
        "best_scores": np.asarray(best_scores),
        "diversity": np.asarray(diversity),
        "personal_best": personal_best.copy(),
        "personal_scores": personal_scores.copy(),
        "best_position": personal_best[best_index].copy(),
        "best_score": float(personal_scores[best_index]),
        "objective_evaluations": n_particles * (steps + 1),
    }


## Establish that the baseline is valid

Design and run checks that establish:

1. the objective has its known minimum value at $(1.3,-0.8)$;
2. a fixed seed reproduces the same run;
3. positions remain within the declared bounds;
4. a newly visited better position replaces the relevant personal-best record; and
5. a run with $N$ particles and $K$ updates makes $N(K+1)$ objective evaluations.

Record what each check tests and why its result supports the implementation. Add any further check required by your investigation.


In [3]:
# Write and run the validation checks here.


# Investigate


## Design the shared-information experiment

Investigate how `shared_weight` affects:

- **reliability:** the fraction of repeated runs reaching a success criterion you define
- **solution quality:** the distribution of final best objective values
- **search concentration:** the distribution of final swarm diversity,

$$
D(n)=\frac{1}{N}\sum_{i=1}^{N}\lVert\mathbf{x}_i^n-\bar{\mathbf{x}}^n\rVert.
$$

A small $D(n)$ means that particles occupy a concentrated region. Choose the success criterion before running the comparison.

| Decision | Your choice and justification |
|---|---|
| prediction |  |
| shared-information weights |  |
| fixed PSO settings |  |
| success criterion |  |
| number of repeated seeds |  |
| outputs and summaries |  |

Vary only `shared_weight`. State the fixed settings and reuse the same seeds in every condition.


### Supplied ensemble helper

The helper runs one condition across a declared collection of seeds. Choose the conditions and construct the comparison yourself.


In [4]:
def run_ensemble(shared_weight, seeds, **fixed_settings):
    runs = [
        run_pso(seed=int(seed), shared_weight=shared_weight, **fixed_settings)
        for seed in seeds
    ]
    return {
        "final_best": np.array([run["best_scores"][-1] for run in runs]),
        "final_diversity": np.array([run["diversity"][-1] for run in runs]),
        "evaluations": np.array([run["objective_evaluations"] for run in runs]),
        "runs": runs,
    }


In [5]:
# Choose the shared-information weights, repeated seeds and fixed settings.
# Run the ensembles, summarise reliability and diversity, and produce a comparison.


## Evidence checkpoint

Report the experimental settings and success criterion, then compare:

- the success fraction
- the distribution of final best objective values
- the distribution of final swarm diversity.

Include one figure that permits a direct comparison. State whether greater concentration coincided with better solutions and give one conclusion limited to the settings tested.


# Optional transfer


## Design a PSO problem

Choose a problem that could be represented as a search through a continuous solution space. You do not need to find or clean a dataset. Specify:

1. the candidate solution and its feasible bounds;
2. the objective function;
3. what would count as a good solution;
4. whether the true global optimum is known;
5. where local optima might arise;
6. the measurements you would use to assess PSO; and
7. a stopping condition.

Explain how each modelling choice represents the original problem. The main task is the specification, not an implementation.

### Optional implementation

If the objective can be evaluated directly, implement it below and test it with the supplied PSO engine. Check the returned solution independently where possible and use repeated runs if the search is stochastic.


In [ ]:
# Optional: define the objective, bounds and PSO settings for your problem.
# Run repeated searches and check the best result independently where possible.


# Exit


## Investigation record

Before leaving, record the strongest conclusion supported by your evidence, its main limitation and the next comparison you would make.
